# Análise Exploratória e Estatística — Students Performance

Este notebook consolida dois trabalhos de análise de dados realizados sobre a base **StudentsPerformance.csv**.  
A proposta é organizar as contribuições dos estudantes em um único relatório, mantendo as análises mais relevantes.

## Objetivos da análise

- Carregar, organizar e compreender a estrutura da base de dados;
- Verificar qualidade dos dados, valores ausentes e duplicidades;
- Analisar a distribuição das variáveis categóricas;
- Comparar o desempenho dos alunos nas notas de Matemática, Leitura e Escrita;
- Investigar relações entre desempenho e variáveis como gênero, almoço, curso de preparação, escolaridade familiar e raça/etnia;
- Aplicar estatísticas descritivas e testes inferenciais;
- Produzir visualizações gráficas úteis para apresentação acadêmica.

## 1. Importação das bibliotecas

As importações foram reunidas em uma única célula para evitar duplicidade.  
Foram mantidas as bibliotecas usadas nos dois trabalhos: `pandas`, `numpy`, `matplotlib`, `seaborn` e funções estatísticas do `scipy`.

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import seaborn as srn
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import norm, chi2_contingency, f_oneway

pd.set_option("display.max_columns", None)
srn.set_theme(style="whitegrid")

## 2. Carregamento da base de dados

A base utilizada é o arquivo `StudentsPerformance.csv`.  
Para executar o notebook sem erro, mantenha esse arquivo na mesma pasta do notebook.

In [ ]:
# Caminho esperado da base de dados
arquivo = "StudentsPerformance.csv"

if not os.path.exists(arquivo):
    raise FileNotFoundError(
        "Arquivo 'StudentsPerformance.csv' não encontrado. "
        "Coloque o CSV na mesma pasta deste notebook antes de executar."
    )

df = pd.read_csv(arquivo, sep=",")

display(df.head())
print("Dimensão original da base:", df.shape)
print("Colunas originais:")
print(df.columns.tolist())

## 3. Pré-processamento e padronização dos dados

Nesta etapa, os nomes das colunas foram traduzidos para português, seguindo a lógica presente nos trabalhos.  
Também foi criada a coluna `media geral`, que representa a média das três notas de cada aluno.

In [ ]:
# Renomeando colunas para facilitar a leitura da análise
df.columns = [
    "genero",
    "raca/etnia",
    "nivel da educacao familiar",
    "almoco",
    "curso de preparacao",
    "nota em matematica",
    "nota em leitura",
    "nota em escrita"
]

colunas_categoricas = [
    "genero",
    "raca/etnia",
    "nivel da educacao familiar",
    "almoco",
    "curso de preparacao"
]

colunas_notas = [
    "nota em matematica",
    "nota em leitura",
    "nota em escrita"
]

# Criando média geral do aluno
df["media geral"] = df[colunas_notas].mean(axis=1)

display(df.head())

## 4. Verificação da qualidade da base

Aqui verificamos tipos de dados, valores nulos e linhas duplicadas.  
Essa etapa consolida a análise inicial feita nos dois arquivos.

In [ ]:
resumo_qualidade = pd.DataFrame({
    "valores_nulos": df.isnull().sum(),
    "valores_unicos": df.nunique()
})

display(resumo_qualidade)
print("Quantidade de linhas duplicadas:", df.duplicated().sum())

## 5. Descrição da base de dados

A base possui variáveis categóricas relacionadas ao perfil dos alunos e variáveis numéricas relacionadas ao desempenho em provas.

- **Variáveis categóricas:** gênero, raça/etnia, nível de educação familiar, tipo de almoço e curso de preparação.
- **Variáveis numéricas:** nota em Matemática, nota em Leitura, nota em Escrita e média geral.

In [ ]:
print("Quantidade de linhas e colunas:", df.shape)

for coluna in colunas_categoricas:
    print(f"\nDistribuição da variável: {coluna}")
    display(df[coluna].value_counts().to_frame("quantidade"))

## 6. Análise exploratória das variáveis categóricas

Os gráficos abaixo mostram a distribuição dos principais grupos presentes na base.  
Foram mantidas as visualizações de distribuição por gênero, raça/etnia, educação familiar, almoço e curso de preparação.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, coluna in enumerate(colunas_categoricas):
    contagem = df[coluna].value_counts()

    if contagem.shape[0] <= 2:
        axes[i].pie(
            contagem.values,
            labels=contagem.index,
            autopct="%1.1f%%",
            startangle=90
        )
        axes[i].set_title(f"Distribuição: {coluna}")
        axes[i].axis("equal")
    else:
        srn.barplot(x=contagem.index, y=contagem.values, ax=axes[i])
        axes[i].set_title(f"Distribuição: {coluna}")
        axes[i].set_xlabel(coluna)
        axes[i].set_ylabel("Quantidade")
        axes[i].tick_params(axis="x", rotation=45)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 7. Estatísticas descritivas das notas

Esta seção reúne médias, medianas, quartis, desvio padrão e outras medidas para as três notas e para a média geral.

In [ ]:
estatisticas_notas = df[colunas_notas + ["media geral"]].describe().T
estatisticas_notas["mediana"] = df[colunas_notas + ["media geral"]].median()
estatisticas_notas["moda"] = df[colunas_notas + ["media geral"]].mode().iloc[0]
estatisticas_notas["variancia"] = df[colunas_notas + ["media geral"]].var(ddof=1)

display(estatisticas_notas.round(2))

media_notas = df[colunas_notas].mean().sort_values(ascending=False)
print("Médias das notas por disciplina:")
display(media_notas.to_frame("media").round(2))

### Interpretação inicial

As medidas de centralidade mostram o desempenho típico dos alunos em cada área avaliada.  
O desvio padrão indica o quanto as notas variam em relação à média: quanto maior o desvio padrão, maior a dispersão das notas.

## 8. Distribuição das notas e boxplots

Os histogramas ajudam a observar o formato da distribuição das notas.  
Os boxplots permitem comparar mediana, amplitude interquartil e possíveis outliers.

In [ ]:
for coluna in colunas_notas + ["media geral"]:
    plt.figure(figsize=(8, 5))
    srn.histplot(df[coluna], kde=True)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.show()

plt.figure(figsize=(9, 5))
srn.boxplot(data=df[colunas_notas + ["media geral"]])
plt.title("Boxplot das notas dos alunos")
plt.ylabel("Nota")
plt.xticks(rotation=20)
plt.show()

## 9. Análise de quartis e outliers da média geral

Esta etapa preserva a lógica do primeiro trabalho para identificar limites pelo método do intervalo interquartil.

In [ ]:
q1 = df["media geral"].quantile(0.25)
q2 = df["media geral"].quantile(0.50)
q3 = df["media geral"].quantile(0.75)

iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = df[(df["media geral"] < limite_inferior) | (df["media geral"] > limite_superior)]

print(f"Q1: {q1:.2f}")
print(f"Mediana: {q2:.2f}")
print(f"Q3: {q3:.2f}")
print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print("Quantidade de outliers:", len(outliers))

display(outliers.sort_values("media geral").head())

## 10. Correlação entre as notas

Esta análise mostra se alunos com bom desempenho em uma prova também tendem a ter bom desempenho nas outras.

In [ ]:
correlacao = df[colunas_notas].corr()

display(correlacao.round(2))

plt.figure(figsize=(8, 5))
srn.heatmap(correlacao, annot=True, cmap="Blues", vmin=0, vmax=1)
plt.title("Correlação entre as notas")
plt.show()

## 11. Médias das notas por grupos

Esta seção consolida as comparações feitas por gênero, almoço, curso de preparação, nível de educação familiar e raça/etnia.

In [ ]:
def medias_por_grupo(coluna):
    resultado = (
        df.groupby(coluna)[colunas_notas + ["media geral"]]
        .mean()
        .sort_values("media geral", ascending=False)
        .round(2)
    )
    print(f"\nMédias por: {coluna}")
    display(resultado)
    return resultado

medias_genero = medias_por_grupo("genero")
medias_almoco = medias_por_grupo("almoco")
medias_curso = medias_por_grupo("curso de preparacao")
medias_educacao = medias_por_grupo("nivel da educacao familiar")
medias_etnia = medias_por_grupo("raca/etnia")

## 12. Visualização das médias por grupos

Foram mantidos gráficos que complementam a leitura das tabelas, priorizando a comparação da `media geral` para evitar excesso de visualizações repetidas.

In [ ]:
variaveis_grupo = [
    "genero",
    "almoco",
    "curso de preparacao",
    "nivel da educacao familiar",
    "raca/etnia"
]

for coluna in variaveis_grupo:
    media_grupo = (
        df.groupby(coluna)["media geral"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    plt.figure(figsize=(10, 5))
    srn.barplot(data=media_grupo, x=coluna, y="media geral")
    plt.title(f"Média geral por {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Média geral")
    plt.ylim(0, 100)
    plt.xticks(rotation=45)

    for i, valor in enumerate(media_grupo["media geral"]):
        plt.text(i, valor + 1, f"{valor:.2f}", ha="center")

    plt.tight_layout()
    plt.show()

## 13. Boxplots com intervalo de confiança da média

O segundo trabalho trouxe uma boa ideia de combinar boxplot com intervalo de confiança de 95%.  
Abaixo, essa ideia foi reorganizada em uma função para evitar repetição de código.

In [ ]:
def boxplot_com_ic(coluna_grupo, coluna_nota, ordem=None):
    medias = df.groupby(coluna_grupo)[coluna_nota].mean()
    desvios = df.groupby(coluna_grupo)[coluna_nota].std(ddof=1)
    tamanhos = df.groupby(coluna_grupo)[coluna_nota].count()

    erro = 1.96 * desvios / np.sqrt(tamanhos)

    if ordem is None:
        ordem = medias.sort_values(ascending=False).index.tolist()

    plt.figure(figsize=(10, 5))
    ax = srn.boxplot(data=df, x=coluna_grupo, y=coluna_nota, order=ordem)

    for i, grupo in enumerate(ordem):
        ax.errorbar(
            x=i,
            y=medias[grupo],
            yerr=erro[grupo],
            fmt="o",
            capsize=8,
            color="black"
        )

    plt.title(f"{coluna_nota} por {coluna_grupo} com IC de 95%")
    plt.xlabel(coluna_grupo)
    plt.ylabel(coluna_nota)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

ordem_educacao = [
    "some high school",
    "high school",
    "some college",
    "associate's degree",
    "bachelor's degree",
    "master's degree"
]

boxplot_com_ic("genero", "media geral", ordem=["female", "male"])
boxplot_com_ic("almoco", "media geral", ordem=["standard", "free/reduced"])
boxplot_com_ic("curso de preparacao", "media geral", ordem=["completed", "none"])
boxplot_com_ic("nivel da educacao familiar", "media geral", ordem=ordem_educacao)

## 14. Relações entre variáveis categóricas e desempenho

Aqui foram preservadas as comparações cruzadas do segundo trabalho, mas compactadas para reduzir repetição.  
O foco passa a ser a `media geral`, em vez de repetir o mesmo conjunto de gráficos para Matemática, Leitura e Escrita separadamente.

In [ ]:
comparacoes_cruzadas = [
    ("nivel da educacao familiar", "curso de preparacao", ordem_educacao),
    ("raca/etnia", "curso de preparacao", ["group A", "group B", "group C", "group D", "group E"]),
    ("almoco", "curso de preparacao", ["standard", "free/reduced"])
]

for eixo_x, hue, ordem in comparacoes_cruzadas:
    plt.figure(figsize=(12, 6))
    srn.boxplot(
        data=df,
        x=eixo_x,
        y="media geral",
        hue=hue,
        order=ordem
    )
    plt.title(f"Média geral por {eixo_x}, separada por {hue}")
    plt.xlabel(eixo_x)
    plt.ylabel("Média geral")
    plt.xticks(rotation=45)
    plt.legend(title=hue)
    plt.tight_layout()
    plt.show()

## 15. Testes de normalidade das notas

Foram mantidos os histogramas, gráficos Q-Q e testes de Shapiro presentes no segundo trabalho.  
Como a base possui 1000 linhas, o teste de Shapiro pode ser sensível: valores de p muito baixos não impedem a análise, mas indicam desvio da normalidade perfeita.

In [ ]:
resultado_normalidade = []

for coluna in colunas_notas:
    plt.figure(figsize=(8, 5))
    srn.histplot(df[coluna], kde=True)
    plt.title(f"Histograma de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 5))
    stats.probplot(df[coluna], fit=True, plot=ax)
    ax.set_title(f"Gráfico Q-Q de {coluna}")
    plt.show()

    shapiro_stat, shapiro_p = stats.shapiro(df[coluna])
    resultado_normalidade.append({
        "variavel": coluna,
        "estatistica_shapiro": shapiro_stat,
        "p_valor": shapiro_p
    })

resultado_normalidade = pd.DataFrame(resultado_normalidade)
display(resultado_normalidade.round(5))

## 16. Probabilidades aproximadas por faixas de quartis

Esta parte preserva a ideia do segundo trabalho de usar a distribuição normal para estimar probabilidades entre faixas de quartis.  
A interpretação deve ser feita como aproximação estatística.

In [ ]:
def probabilidades_quartis(coluna):
    media = df[coluna].mean()
    desvio = df[coluna].std(ddof=1)
    quartis = np.quantile(df[coluna], [0, 0.25, 0.50, 0.75, 1])

    faixas = []
    acumuladas = norm.cdf(quartis, media, desvio)

    for i in range(1, len(quartis)):
        prob = acumuladas[i] - acumuladas[i - 1]
        faixas.append({
            "variavel": coluna,
            "faixa": f"{quartis[i-1]:.2f} até {quartis[i]:.2f}",
            "probabilidade_aproximada": prob
        })

    return pd.DataFrame(faixas)

probabilidades = pd.concat(
    [probabilidades_quartis(coluna) for coluna in colunas_notas],
    ignore_index=True
)

display(probabilidades.round(3))

## 17. Testes estatísticos inferenciais

Nesta etapa foram reunidos os testes estatísticos complementares dos trabalhos:

- **Teste t de Student:** compara a média geral entre alunos que completaram e não completaram o curso de preparação.
- **ANOVA:** verifica se há diferença significativa de média geral entre níveis de educação familiar.
- **Qui-quadrado:** verifica associação entre variáveis categóricas.

In [ ]:
# Teste t: curso de preparação e média geral
grupo_completo = df[df["curso de preparacao"] == "completed"]["media geral"]
grupo_nao_completo = df[df["curso de preparacao"] == "none"]["media geral"]

t_stat, p_valor_t = stats.ttest_ind(grupo_completo, grupo_nao_completo, equal_var=False)

print("Teste t de Student — média geral por curso de preparação")
print(f"Estatística t: {t_stat:.4f}")
print(f"p-valor: {p_valor_t:.6f}")

if p_valor_t < 0.05:
    print("Resultado: rejeitamos H0. Existe diferença significativa entre os grupos.")
else:
    print("Resultado: não rejeitamos H0. Não há evidência de diferença significativa entre os grupos.")

In [ ]:
# ANOVA: nível de educação familiar e média geral
grupos_educacao = [
    grupo["media geral"].values
    for _, grupo in df.groupby("nivel da educacao familiar")
]

f_stat, p_valor_anova = f_oneway(*grupos_educacao)

print("ANOVA — média geral por nível de educação familiar")
print(f"Estatística F: {f_stat:.4f}")
print(f"p-valor: {p_valor_anova:.6f}")

if p_valor_anova < 0.05:
    print("Resultado: rejeitamos H0. Existe diferença significativa entre pelo menos dois níveis de escolaridade.")
else:
    print("Resultado: não rejeitamos H0. Não há evidência de diferença significativa entre os níveis.")

In [ ]:
# Testes Qui-Quadrado entre variáveis categóricas
pares_quiquadrado = [
    ("curso de preparacao", "nivel da educacao familiar"),
    ("curso de preparacao", "raca/etnia"),
    ("nivel da educacao familiar", "raca/etnia"),
    ("almoco", "raca/etnia")
]

resultados_quiquadrado = []

for var1, var2 in pares_quiquadrado:
    tabela = pd.crosstab(df[var1], df[var2])
    chi2, p, gl, expected = chi2_contingency(tabela)

    resultados_quiquadrado.append({
        "variavel_1": var1,
        "variavel_2": var2,
        "qui_quadrado": chi2,
        "graus_de_liberdade": gl,
        "p_valor": p,
        "interpretacao": "dependentes/associadas" if p < 0.05 else "independentes"
    })

resultados_quiquadrado = pd.DataFrame(resultados_quiquadrado)
display(resultados_quiquadrado.round(5))

## 18. Principais descobertas

Com base nas análises realizadas, podemos destacar:

1. A base possui 1000 registros e 8 colunas originais, além da coluna criada `media geral`.
2. A maior parte dos alunos não realizou curso de preparação.
3. Alunos que completaram o curso de preparação apresentaram desempenho médio superior.
4. A variável `almoco` também apresenta diferença relevante nas médias, com maior desempenho médio para alunos com almoço `standard`.
5. As notas de Leitura e Escrita apresentaram correlação muito forte.
6. O grupo `female` teve maiores médias em Leitura e Escrita, enquanto o grupo `male` teve maior média em Matemática.
7. Níveis mais altos de educação familiar aparecem associados a médias gerais mais altas.
8. As análises inferenciais indicam que algumas diferenças observadas entre grupos são estatisticamente relevantes.